In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


# MLOps Group Project - Experiment 1: Baseline Fine-Tuning
**Task 3 & Task 4: Model Selection, Kaggle Training, and W&B Tracking**


## 1. Environment Setup  
To build our end-to-end pipeline, we need several libraries: transformers and datasets for handling the Hugging Face model and data, evaluate to calculate our accuracy metrics, and wandb to track our experiment parameters and metrics securely.

In [2]:
!pip install -q transformers datasets evaluate wandb scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


## 2. Secure Authentication
**Justification:** Hardcoding API keys is a bad MLOps practice. We are using `Kaggle Secrets` to securely fetch our Weights & Biases (W&B) and Hugging Face tokens. W&B will act as our central dashboard for tracking loss and accuracy, while the Hugging Face login prepares our environment for Task 5 (Pushing the model to the Hub).

In [3]:
import os
import wandb
import warnings
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

warnings.filterwarnings("ignore", category=SyntaxWarning)

print("Fetching secrets from Kaggle...")
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY") 
hf_token = user_secrets.get_secret("HF_TOKEN")       

# Weights & Biases Authentication & Validation
print("\nConnecting to Weights & Biases...")
os.environ["WANDB_API_KEY"] = wandb_key
wandb_success = wandb.login()

if wandb_success:
    print("✅ W&B Connection Established Successfully!")
else:
    print("❌ W&B Connection Failed. Please check your WANDB_API_KEY.")

os.environ["WANDB_PROJECT"] = "mlops-emotion-classification" 
os.environ["WANDB_LOG_MODEL"] = "checkpoint" 

# Hugging Face Authentication & Validation
print("\nConnecting to Hugging Face...")
login(token=hf_token)

try:
    # whoami() actively pings Hugging Face to verify the token works
    hf_user_info = whoami()
    print(f"✅ Hugging Face Connection Established Successfully! Logged in as: {hf_user_info['name']}")
except Exception as e:
    print(f"❌ Hugging Face Connection Failed. Please check your HF_TOKEN. Error: {e}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


Fetching secrets from Kaggle...

Connecting to Weights & Biases...


wandb: Currently logged in as: zeeshu-irritant (zeeshu-irritant-prom-iit-rajasthan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B Connection Established Successfully!

Connecting to Hugging Face...
✅ Hugging Face Connection Established Successfully! Logged in as: zeeshan-hf


## 3. Data Loading & Model Selection (Task 3)

**Justification:** 
* **Model Selection:** We selected `distilbert-base-uncased`. As per the Hugging Face model card, DistilBERT is a smaller, faster, and lighter version of BERT. At roughly 260MB (uncompressed), it is highly compact, meaning it trains quickly and fits easily within Kaggle's free T4 GPU memory limits, fulfilling the assignment's "compact model" constraint.
* **Data Preparation:** We are using the `dair-ai/emotion` dataset. To iterate quickly and avoid exhausting Kaggle quotas, we are taking a random, seeded subset of 8,000 training samples and 1,500 validation samples. We are also explicitly setting `num_labels=6` to match our `id2label.json` mapping.

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading dataset...")
# Load the Emotion Dataset
dataset = load_dataset("dair-ai/emotion")

# Select compact model
model_name = "distilbert-base-uncased"
print(f"Loading Tokenizer and Model: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
# We set num_labels=6 because our id2label.json has 6 emotion classes
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

print("Tokenizing data...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Downsampling for faster experimentation
print("Downsampling for Experiment 1...")
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(8000))
small_eval = tokenized_datasets["validation"].shuffle(seed=42).select(range(1500))

print(f"✅ Data Preparation Complete! Training samples: {len(small_train)} | Validation samples: {len(small_eval)}")

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

...

Tokenizing data...


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Downsampling for Experiment 1...
✅ Data Preparation Complete! Training samples: 8000 | Validation samples: 1500


## 4. Experiment 1: Baseline Training (Task 4)

**Justification:** For our first version, we are establishing a baseline performance metric. We will use standard hyperparameters for DistilBERT fine-tuning:
* **Learning Rate:** `5e-5` (standard for fine-tuning without catastrophic forgetting)
* **Batch Size:** `16` (fits comfortably in T4 GPU memory)
* **Epochs:** `3` (enough to see convergence on a 5k sample without overfitting)
* **Evaluation & Saving:** Strategy set to `epoch` to monitor validation loss.

We explicitly link this run to our Weights & Biases dashboard by setting `report_to="wandb"` and giving it a descriptive run name (`distilbert-exp1-baseline`) so we can easily compare it against Version 2 later.

In [5]:
!pip install -q evaluate

import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

print("Loading evaluation metrics...")
# Define Accuracy Metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    from sklearn.metrics import f1_score
    results = metric.compute(predictions=predictions, references=labels)
    results["f1"] = f1_score(labels, predictions, average="weighted")
    return results

print("Configuring Training Arguments for Experiment 1...")
# Baseline Hyperparameters (Experiment 1)
training_args = TrainingArguments(
    output_dir="./results_exp1",
    learning_rate=5e-5,               
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,               
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",                
    run_name="distilbert-nik-Exp-1",
    push_to_hub=False                 
)

print("Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

print("Starting Experiment 1 Training _Nikunj...")
# Execute Training
trainer.train()

print("Training complete! Syncing final logs to Weights & Biases...")
# Close the W&B run to finalize logs
import wandb
wandb.finish()

print("✅ Experiment 1 Finished! Click the wandb link above to view your dashboard.")

Loading evaluation metrics...


Configuring Training Arguments for Experiment 1...
Initializing Trainer...
Starting Experiment 1 Training _Nikunj...


wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260614_111940-4rfkctgh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run distilbert-nik-Exp-1
wandb: ⭐️ View project at https://wandb.ai/zeeshu-irritant-prom-iit-rajasthan/mlops-emotion-classification
wandb: 🚀 View run at https://wandb.ai/zeeshu-irritant-prom-iit-rajasthan/mlops-emotion-classification/runs/4rfkctgh
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.654390,0.494795,0.915333
2,0.301808,0.336015,0.932667
3,0.185520,0.329110,0.933333


wandb: WARNING Artifact "model-distilbert-nik-Exp-1" already exists with the same content. No new version will be created.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: uploading artifact model-distilbert-nik-Exp-1; updating run metadata


Training complete! Syncing final logs to Weights & Biases...


wandb: uploading artifact model-distilbert-nik-Exp-1
wandb: uploading wandb-summary.json; uploading config.yaml; uploading output.log
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁██
wandb:               eval/loss █▁▁
wandb:            eval/runtime ▁█▂
wandb: eval/samples_per_second █▁▇
wandb:   eval/steps_per_second █▁▇
wandb:             train/epoch ▁▂▂▃▃▃▄▄▅▅▅▅▆▇▇▇███
wandb:       train/global_step ▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇███
wandb:         train/grad_norm ▁▄▄▂▂▁▄▂█▄▄▂▃▄█
wandb:     train/learning_rate █▇▇▇▆▆▅▄▄▃▃▂▂▁▁
wandb:              train/loss █▅▃▂▂▂▂▂▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.93333
wandb:               eval/loss 0.32911
wandb:            eval/runtime 6.4652
wandb: eval/samples_per_second 232.01
wandb:   eval/steps_per_second 7.27
wandb:              total_flos 1589722177536000.0
wandb:             train/epoch 3
wandb:       train/global_step 750
wandb:         train/grad_norm 18.01617
wandb:     train/learning_rate 0.0
wandb:   

✅ Experiment 1 Finished! Click the wandb link above to view your dashboard.
